# 🧙‍♂️ Statistical Bigram Language Model: Interactive Demo & Walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thisiskamaljoshi/simple-bigram-model/blob/main/notebooks/demo.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Python 3.11+](https://img.shields.io/badge/python-3.11+-blue.svg)](https://www.python.org/downloads/)

Welcome to the interactive walkthrough of the **Statistical Bigram Language Model**!

This notebook demonstrates the core foundational concepts of Natural Language Processing (NLP) and statistical language modeling built entirely from first principles:
- **Zero Downloads Required**: Uses an embedded mini-corpus of classic fairy tales for instant, deterministic training in under a second.
- **Runnable Anywhere**: Fully self-contained and pre-configured for **Google Colab** and local Jupyter environments.

---

### What is a Bigram Language Model?
A bigram language model estimates the probability of a word sequence $W = (w_1, w_2, \dots, w_N)$ using the **First-Order Markov Assumption**: the probability of the current word depends **only** on the single preceding word:

$$P(w_1, w_2, \dots, w_N) = \prod_{i=1}^{N} P(w_i \mid w_{i-1})$$

In this notebook, we walk step-by-step through the 4 essential stages:
1. **Tokenization & Sentence Boundary Injection**: Isolating punctuation and adding `⟨START⟩` / `⟨END⟩` states.
2. **Frequency Transition Matrix (MLE)**: Counting adjacent word pairs and computing transition probabilities.
3. **Temperature-Scaled Sampling**: Comparing Greedy ($T=0.01$), Balanced ($T=0.7$), and Creative ($T=1.5$) generation with an interactive slider.
4. **Intrinsic Evaluation**: Calculating test set bigram coverage, log-likelihood, and perplexity ($PP$).

In [ ]:
# Setup environment: clone repo if running in Colab / external environment
import os
import sys

# Ensure repository files are in working directory or sys.path
if not os.path.exists("tokenizer.py") and not os.path.exists("../tokenizer.py"):
    print("Cloning repository simple-bigram-model from GitHub...")
    !git clone https://github.com/thisiskamaljoshi/simple-bigram-model.git
    %cd simple-bigram-model
elif os.path.exists("../tokenizer.py") and not os.path.exists("tokenizer.py"):
    # If running from within the notebooks/ subfolder locally
    os.chdir("..")

if "." not in sys.path:
    sys.path.insert(0, ".")

# Import core simple-bigram-model modules
from tokenizer import Tokenizer
from sentenceprocessor import SentenceProcessor
from trainer import Trainer
from bigram import BigramModel
from sampler import Sampler
from generator import Generator
from detokenizer import Detokenizer
from evaluator import Evaluator

# Visualization & helper libraries
import math
import random
import pandas as pd

# Optional interactive widgets and plotting (pre-installed in Google Colab)
try:
    import matplotlib.pyplot as plt
    has_plt = True
except ImportError:
    has_plt = False

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    has_widgets = True
except ImportError:
    has_widgets = False

print("Environment configured successfully!")
print("Working directory:", os.path.abspath("."))

---

## 📚 Embedded Mini-Corpus: Classic Fairy Tales

To make this notebook 100% self-contained with **zero file downloads**, we embed a training corpus of 5 classic fairy tales:
1. *The Three Little Pigs*
2. *Goldilocks and the Three Bears*
3. *Little Red Riding Hood*
4. *The Tortoise and the Hare*
5. *The Frog Prince*

Fairy tales are ideal for learning bigram modeling because they feature:
- Repetitive narrative patterns (*"Once upon a time"*, *"huffed and puffed"*, *"just right"*).
- Shared thematic vocabulary (*"little"*, *"wolf"*, *"cottage"*, *"forest"*).
- Clear dialogue punctuation and sentence boundaries.

In [ ]:
# Embedded training mini-corpus (Classic Fairy Tales)
FAIRY_TALES_TRAIN = """
Once upon a time, there were three little pigs. The first little pig built a house of straw. The second little pig built a house of sticks. The third little pig built a house of bricks. One day, a big bad wolf came to the forest. The wolf walked to the straw house and knocked on the door. "Little pig, little pig, let me come in!" cried the wolf. "Not by the hair on my chinny chin chin!" said the little pig. So the big wolf huffed and puffed, and he blew the house down. The little pig ran quickly to the house of sticks. The big wolf came and huffed and puffed, and he blew the stick house down too. Both little pigs ran fast to the house of bricks. The big bad wolf arrived and tried to blow the brick house down. He huffed and puffed with all his might, but the strong house did not fall. The three little pigs were happy and safe inside.

Once upon a time, there was a little girl named Goldilocks. She went for a walk in the deep forest. Soon, she saw a little cottage. The cottage belonged to three bears: Papa Bear, Mama Bear, and Baby Bear. On the table stood three bowls of porridge. Goldilocks tasted the porridge from the great big bowl, but it was too hot. She tasted the porridge from the medium bowl, but it was too cold. Then she tasted the little bowl, and it was just right. She ate it all up! Next, she tried three chairs. The great big chair was too hard, the medium chair was too soft, but the little chair was just right. Suddenly, the chair broke! Goldilocks felt tired and went upstairs. She lay down in the tiny bed and fell fast asleep. Soon the three bears came home. "Someone has been eating my porridge!" said Papa Bear. "Someone has been sitting in my chair!" said Baby Bear. When Goldilocks woke up and saw the bears, she jumped out of bed and ran away into the forest.

Once upon a time, a kind girl named Little Red Riding Hood lived near the forest. One sunny morning, her mother baked fresh bread and cakes. "Take this basket of food to your grandmother," her mother said. "Stay on the path and do not speak to strangers." Little Red Riding Hood walked through the woods. Along the path, she met a clever wolf. "Where are you going, little girl?" asked the wolf politely. "I am going to grandmother's cottage," she answered. The sly wolf ran ahead and arrived at the cottage first. He knocked on the door and hid inside. When Little Red Riding Hood arrived, she noticed something strange. "Grandmother, what big eyes you have!" she said. "All the better to see you with, my dear," replied the wolf. "Grandmother, what big ears you have!" she said. "All the better to hear you with, my dear!" "Grandmother, what big teeth you have!" cried the girl. "All the better to eat you with!" roared the wolf. Just then, a brave woodsman heard the noise, rushed inside, and chased the wolf away forever.

A boastful hare and a quiet tortoise decided to run a race through the sunny valley. The hare laughed proudly at the slow tortoise. "You will never win!" cried the speedy hare. The race began, and the hare darted ahead quickly. Seeing how far behind the tortoise was, the confident hare lay down under a shady tree to take a nap. Meanwhile, the tortoise kept walking step by step, never stopping to rest. Slowly and steadily, the patient tortoise passed the sleeping hare. When the hare woke up, he ran as fast as the wind, but it was too late. The tortoise had already crossed the finish line! All the animals cheered with joy.

Once upon a time, a beautiful princess played in the royal garden with her favorite golden ball. Suddenly, the golden ball bounced high and fell into a deep, dark well. The princess cried bitterly. A friendly green frog poked his head out of the water. "Why are you crying, little princess?" asked the frog. "My golden ball has fallen into the well," she wept. "If you promise to be my friend, I will bring it back to you," said the frog. The princess agreed, and the frog dove deep down to fetch the golden ball. She was overjoyed, took the ball, and ran back to the castle. That evening, the frog knocked on the palace door. The wise king said, "A promise made must be kept." So the princess invited the frog to dine with her. As kindness filled her heart, a wonderful enchantment broke, and the little frog transformed into a handsome prince.
""".strip()

# Test Corpus 1: In-domain sentences with 100% seen transitions (to demonstrate finite perplexity)
TEST_CORPUS_SEEN = """
Once upon a time, there were three little pigs. The big bad wolf arrived and tried to blow the brick house down.
""".strip()

# Test Corpus 2: In-domain held-out fairy tale sentences with some unseen transitions
TEST_CORPUS_HELD_OUT = """
Once upon a time, the clever wolf knocked on the cottage door. The three little pigs were safe inside the house.
""".strip()

# Test Corpus 3: Out-of-domain technical text with unseen vocabulary and transitions
TEST_CORPUS_OUT_OF_DOMAIN = """
Quantum computing hardware executes deep tensor algorithms using probabilistic superposition.
""".strip()

print(f"Training corpus loaded: {len(FAIRY_TALES_TRAIN):,} characters, {len(FAIRY_TALES_TRAIN.split()):,} words.")

---

## 🔍 Step 1: Tokenization and Boundary Injection Visualization

Before training a language model, raw text must be converted into a discrete sequence of tokens.

### 1. Punctuation-Aware Tokenization (`Tokenizer`)
Simple whitespace splitting (`text.split()`) fails because punctuation sticks to words (e.g. `"pigs."` vs `"pigs"`).
Our custom [`Tokenizer`](file:///tokenizer.py) cleanly isolates punctuation marks as independent tokens so `"wolf,"`, `"wolf."`, and `"wolf"` all share the root vocabulary word `"wolf"`.

### 2. Sentence Boundary Injection (`SentenceProcessor`)
In a generative Markov chain:
- How does the model know how to start a sentence? $ightarrow$ We inject a special `⟨START⟩` token at the start of every sentence.
- How does the model know when to stop generating? $ightarrow$ We inject a special `⟨END⟩` token at every sentence termination (`.`, `!`, `?`).

Let's visualize this transformation on a sample excerpt!

In [ ]:
# Initialize tokenization components
tokenizer = Tokenizer()
processor = SentenceProcessor()

sample_text = 'Once upon a time, there were three little pigs! "Little pig, let me come in!" cried the wolf.'

# 1. Raw tokenization
raw_tokens = tokenizer.tokenize(sample_text)

# 2. Sentence boundary injection
boundary_tokens = processor.process(raw_tokens)

print("=" * 70)
print("ORIGINAL TEXT:")
print(f"  {sample_text}")
print("=" * 70)

print(f"\n1. RAW TOKENS ({len(raw_tokens)} tokens):")
print("  " + " | ".join(raw_tokens[:16]) + " ...")

print(f"\n2. BOUNDARY-INJECTED TOKENS ({len(boundary_tokens)} tokens):")
print("  " + " | ".join(boundary_tokens[:20]) + " ...")

# Formatted DataFrame comparison of first sentence
sample_df = pd.DataFrame({
    "Step": ["1. Raw Tokens", "2. Boundary Tokens"],
    "Sequence": [
        " ".join(raw_tokens[:11]),
        " ".join(boundary_tokens[:13])
    ]
})
display(sample_df)

In [ ]:
# Process the full training corpus
train_raw_tokens = tokenizer.tokenize(FAIRY_TALES_TRAIN)
train_tokens = processor.process(train_raw_tokens)

print(f"Corpus tokenization summary:")
print(f"  • Raw tokens extracted:       {len(train_raw_tokens):,}")
print(f"  • Boundary-injected tokens:   {len(train_tokens):,}")
print(f"  • Injected <START>/<END> tags: {len(train_tokens) - len(train_raw_tokens):,}")

# Display first 24 tokens of the processed training corpus
print("\nFirst 24 processed training tokens:")
print(train_tokens[:24])

---

## 📊 Step 2: Frequency Transition Matrix Inspection

Now we train the model by counting adjacent token pairs and converting counts into transition probabilities using **Maximum Likelihood Estimation (MLE)**.

### Mathematical Formulation
Given adjacent tokens $(w_{i-1}, w_i)$, the conditional probability of $w_i$ following $w_{i-1}$ is:

$$P(w_i \mid w_{i-1}) = rac{	ext{Count}(w_{i-1}, w_i)}{\sum_{w'} 	ext{Count}(w_{i-1}, w')} = rac{	ext{Count}(w_{i-1}, w_i)}{	ext{Count}(w_{i-1})}$$

Every condition token $w_{i-1}$ defines a valid probability distribution over all possible next tokens $w_i$:

$$\sum_{w'} P(w' \mid w_{i-1}) = 1.0$$

In [ ]:
trainer = Trainer()
bigram_model = BigramModel()
evaluator = Evaluator()

# Count bigram pairs
bigram_counts = trainer.train(train_tokens)

# Compute MLE transition probabilities
probabilities = bigram_model.train(bigram_counts)

# Model summary statistics
vocab_size = evaluator.vocabulary_size(probabilities)
num_bigrams = evaluator.unique_bigram_count(probabilities)

print(f"Training completed successfully!")
print(f"  • Unique vocabulary size: {vocab_size:,} words")
print(f"  • Unique bigrams learned: {num_bigrams:,} transitions")

In [ ]:
def inspect_transitions(word: str, top_n: int = 8) -> pd.DataFrame:
    """Displays top transitions for a given conditioning word as a formatted DataFrame."""
    if word not in probabilities:
        print(f"Word '{word}' not in model vocabulary.")
        return pd.DataFrame()
    
    transitions = probabilities[word]
    counts = bigram_counts[word]
    total_count = sum(counts.values())
    
    rows = []
    for next_word, prob in sorted(transitions.items(), key=lambda x: x[1], reverse=True)[:top_n]:
        rows.append({
            "Given Token (w_i-1)": word,
            "Next Token (w_i)": next_word,
            "Count": counts[next_word],
            "Total Preceding": total_count,
            "Probability P(w_i | w_i-1)": f"{prob:.4f}",
            "Percent": f"{prob * 100:.2f}%"
        })
    return pd.DataFrame(rows)

print("Top transitions following '<START>' (how sentences begin):")
display(inspect_transitions("<START>", top_n=6))

print("\nTop transitions following 'little':")
display(inspect_transitions("little", top_n=6))

print("\nTop transitions following 'wolf':")
display(inspect_transitions("wolf", top_n=6))

In [ ]:
# Validate probability axioms: each condition word's transitions must sum to 1.0
evaluator.validate_probabilities(probabilities)
print("[OK] Probability validation passed: all probability rows sum to 1.0 (+/- 0.001)")


# Visualize transition matrix heatmap for selected frequent words
selected_words = ["<START>", "Once", "the", "little", "pig", "wolf", "house", "straw", "bears", "forest", ".", "<END>"]

# Build matrix of probabilities between selected words
matrix_data = []
for w1 in selected_words:
    row = []
    for w2 in selected_words:
        row.append(probabilities.get(w1, {}).get(w2, 0.0))
    matrix_data.append(row)

matrix_df = pd.DataFrame(matrix_data, index=selected_words, columns=selected_words)

if has_plt:
    plt.figure(figsize=(10, 8))
    plt.imshow(matrix_df.values, cmap="YlGnBu", interpolation="nearest")
    plt.title("Bigram Transition Probability Matrix P(Next | Given)", fontsize=14, pad=15)
    plt.xticks(range(len(selected_words)), selected_words, rotation=45, ha="right", fontsize=10)
    plt.yticks(range(len(selected_words)), selected_words, fontsize=10)
    plt.xlabel("Next Word (w_i)", fontsize=11, labelpad=10)
    plt.ylabel("Given Word (w_i-1)", fontsize=11, labelpad=10)
    plt.colorbar(label="Transition Probability")
    
    # Annotate non-zero cells
    for i in range(len(selected_words)):
        for j in range(len(selected_words)):
            val = matrix_df.iloc[i, j]
            if val > 0.01:
                plt.text(j, i, f"{val:.2f}", ha="center", va="center", 
                         color="white" if val > 0.4 else "black", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("\nTransition Matrix Preview (probabilities):")
    display(matrix_df.round(3))

---

## 🌡️ Step 3: Temperature Scaling & Text Generation

When generating text autoregressively, we pick the next word from the probability distribution $P(w_i \mid w_{i-1})$.
**Temperature scaling** controls the randomness/entropy of sampling:

$$P_T(w_i \mid w_{i-1}) = rac{P(w_i \mid w_{i-1})^{1/T}}{\sum_{w'} P(w' \mid w_{i-1})^{1/T}}$$

### Temperature Regimes:
| Temperature | Name | Behavior | Output Characteristics |
| :--- | :--- | :--- | :--- |
| **$T = 0.01$** | **Greedy / Argmax** | Picks $\operatorname{argmax} P(w_i \mid w_{i-1})$ deterministically | Deterministic, highly repetitive, shortest most likely paths |
| **$T = 0.70$** | **Balanced** | Sharpened sampling distribution | Coherent, diverse, natural storytelling flow |
| **$T = 1.50$** | **Creative / High Entropy** | Flattens probabilities towards uniform distribution | Diverse, surprising, whimsical, occasionally nonsensical |

In [ ]:
def get_temperature_distribution(word: str, temperature: float) -> dict[str, float]:
    """Computes scaled probabilities for a given word and temperature."""
    raw_dist = probabilities.get(word, {})
    if not raw_dist:
        return {}
    if temperature <= 0.05:
        max_word = max(raw_dist, key=raw_dist.get)
        return {w: 1.0 if w == max_word else 0.0 for w in raw_dist}
    scaled = {w: math.exp(math.log(max(p, 1e-12)) / temperature) for w, p in raw_dist.items()}
    total = sum(scaled.values())
    return {w: p / total for w, p in scaled.items()}

# Compare distributions for 'little' across temperatures
test_word = "little"
temps = [0.01, 0.7, 1.0, 1.5]
top_candidates = [w for w, _ in sorted(probabilities[test_word].items(), key=lambda x: x[1], reverse=True)[:6]]

temp_comparison_data = {"Next Token": top_candidates}
for T in temps:
    dist = get_temperature_distribution(test_word, T)
    temp_comparison_data[f"T={T}"] = [dist.get(w, 0.0) for w in top_candidates]

temp_df = pd.DataFrame(temp_comparison_data)

if has_plt:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
    for idx, (T, ax) in enumerate(zip(temps, axes)):
        col_name = f"T={T}"
        ax.bar(temp_df["Next Token"], temp_df[col_name], color=["#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"][idx])
        ax.set_title(f"Temperature T = {T}", fontsize=12, fontweight="bold")
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis='x', rotation=45)
        ax.grid(axis='y', linestyle='--', alpha=0.5)
        if idx == 0:
            ax.set_ylabel("Probability", fontsize=11)
    plt.suptitle(f"Next-Token Probability Distribution following '{test_word}' under Temperature Scaling", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print(f"Probability distribution following '{test_word}':")
    display(temp_df.round(3))

In [ ]:
generator = Generator()

# Generate sentences at 3 distinct temperatures
comparison_scenarios = [
    (0.01, "Greedy (T = 0.01)", "Picks the single highest-probability token at each step. Fast, deterministic, repetitive."),
    (0.70, "Balanced (T = 0.70)", "Balances probability weights with stochastic exploration. Smooth and coherent."),
    (1.50, "Creative (T = 1.50)", "Flattens probabilities, exploring rare bigram transitions. Whimsical and diverse.")
]

print("=" * 80)
print("AUTOREGRESSIVE GENERATION COMPARISON")
print("=" * 80)

for temp, title, explanation in comparison_scenarios:
    print(f"\n🔹 {title.upper()}:")
    print(f"   Explanation: {explanation}")
    print("   Generated Sentences:")
    for seed in [10, 42, 99]:
        random.seed(seed)
        sentence = generator.generate_sentence(probabilities, max_tokens=40, temperature=temp)
        print(f"     • [Seed {seed}]: {sentence}")

In [ ]:
def interactive_generation(temperature: float = 0.7, max_tokens: int = 40, seed: int = 42, num_sentences: int = 3):
    """Interactive generator callback."""
    print(f"--- Generating with Temperature={temperature:.2f} | Seed={seed} ---")
    for i in range(num_sentences):
        random.seed(seed + i * 17)
        sentence = generator.generate_sentence(probabilities, max_tokens=max_tokens, temperature=temperature)
        print(f"[{i+1}] {sentence}")

if has_widgets:
    interact(
        interactive_generation,
        temperature=FloatSlider(min=0.01, max=2.0, step=0.05, value=0.70, description="Temp (T):"),
        max_tokens=IntSlider(min=10, max=80, step=5, value=40, description="Max Tokens:"),
        seed=IntSlider(min=0, max=100, step=1, value=42, description="Random Seed:"),
        num_sentences=IntSlider(min=1, max=5, step=1, value=3, description="Sentences:")
    )
else:
    # Fallback demonstration when widgets are not available
    print("Interactive widgets not active; running sample generation:")
    interactive_generation(temperature=0.7, max_tokens=40, seed=42, num_sentences=3)

---

## 📏 Step 4: Intrinsic Evaluation (Calculating Coverage and Perplexity)

How do we scientifically measure the quality of a statistical language model?
We evaluate on held-out text sequences using two key metrics: **Bigram Coverage** and **Perplexity**.

### 1. Test Set Bigram Coverage
Measures the fraction of test set transitions that were actually observed in the training corpus:

$$\text{Coverage} = \frac{\text{Seen Bigrams}}{\text{Total Evaluated Bigrams}} = \frac{\sum_{i=1}^N \mathbb{I}[ (w_{i-1}, w_i) \in \text{Model} ]}{N}$$

### 2. Log-Probability and Perplexity ($PP$)
The total log-likelihood of a test sequence $W = (w_1, w_2, \dots, w_N)$ is the sum of transition log-probabilities:

$$\text{Total Log-Probability} = \sum_{i=1}^N \ln P(w_i \mid w_{i-1})$$

$$\text{Average Log-Probability } \bar{\ell} = \frac{1}{N} \sum_{i=1}^N \ln P(w_i \mid w_{i-1})$$

**Perplexity** is the exponentiated negative average log-likelihood:

$$PP(W) = \exp(-\bar{\ell}) = \left( \prod_{i=1}^N P(w_i \mid w_{i-1}) \right)^{-\frac{1}{N}}$$

> **Intuition**: Perplexity represents the model's **effective branching factor**. A perplexity of $3.58$ means that at each word step, the model is as uncertain as if choosing uniformly among $\approx 3.6$ words. **Lower perplexity indicates higher confidence and better prediction.**

### ⚠️ The Zero-Frequency Problem
In an unsmoothed MLE model, if **even a single bigram** has never been seen in the training data:

$$P(w_{\text{unseen}} \mid w_{i-1}) = 0.0 \implies \ln(0.0) = -\infty \implies PP = \exp(\infty) = \infty$$

Let's test both seen and unseen scenarios below!

In [ ]:
# 1. Evaluate on in-domain text with 100% seen transitions
test_tokens_seen = processor.process(tokenizer.tokenize(TEST_CORPUS_SEEN))
metrics_seen = evaluator.evaluate(probabilities, test_tokens_seen)

print("=" * 60)
print("EVALUATION 1: IN-DOMAIN TEXT (100% SEEN BIGRAMS)")
print("=" * 60)
evaluator.report(metrics_seen)

# Step-by-step breakdown of individual bigram log-probabilities
seen_bigrams = evaluator.get_bigrams(test_tokens_seen)
breakdown_rows = []
for w1, w2 in seen_bigrams[:10]:
    prob = evaluator.get_probability(probabilities, w1, w2)
    log_p = math.log(prob) if prob > 0 else -math.inf
    breakdown_rows.append({
        "Current (w_i-1)": w1,
        "Next (w_i)": w2,
        "P(w_i | w_i-1)": f"{prob:.4f}",
        "ln P(w_i | w_i-1)": f"{log_p:.4f}"
    })

print("\nStep-by-Step Transition Breakdown (first 10 bigrams):")
display(pd.DataFrame(breakdown_rows))

In [ ]:
# 2. Evaluate on in-domain held-out text (contains some unseen bigrams)
test_tokens_held = processor.process(tokenizer.tokenize(TEST_CORPUS_HELD_OUT))
metrics_held = evaluator.evaluate(probabilities, test_tokens_held)

# 3. Evaluate on out-of-domain technical text (contains mostly unseen bigrams)
test_tokens_ood = processor.process(tokenizer.tokenize(TEST_CORPUS_OUT_OF_DOMAIN))
metrics_ood = evaluator.evaluate(probabilities, test_tokens_ood)

print("=" * 60)
print("EVALUATION 2: HELD-OUT FAIRY TALE (UNSEEN BIGRAMS)")
print("=" * 60)
evaluator.report(metrics_held)

# Identify the exact unseen bigrams
held_bigrams = evaluator.get_bigrams(test_tokens_held)
unseen_list = [f"('{w1}', '{w2}')" for w1, w2 in held_bigrams if evaluator.get_probability(probabilities, w1, w2) == 0.0]
print(f"\nUnseen bigrams ({len(unseen_list)}): " + ", ".join(unseen_list))

print("\n" + "=" * 60)
print("EVALUATION 3: OUT-OF-DOMAIN TEXT (TECHNICAL VOCABULARY)")
print("=" * 60)
evaluator.report(metrics_ood)

In [ ]:
# Comparative Evaluation Summary Table
summary_data = [
    {
        "Corpus": "1. Seen In-Domain",
        "Total Bigrams": metrics_seen["evaluated_bigrams"],
        "Seen": metrics_seen["seen_bigrams"],
        "Unseen": metrics_seen["unseen_bigrams"],
        "Coverage": f"{metrics_seen['coverage']:.1%}",
        "Avg Log-Prob": f"{metrics_seen['average_log_probability']:.3f}",
        "Perplexity": f"{metrics_seen['perplexity']:.2f}"
    },
    {
        "Corpus": "2. Held-Out In-Domain",
        "Total Bigrams": metrics_held["evaluated_bigrams"],
        "Seen": metrics_held["seen_bigrams"],
        "Unseen": metrics_held["unseen_bigrams"],
        "Coverage": f"{metrics_held['coverage']:.1%}",
        "Avg Log-Prob": str(metrics_held["average_log_probability"]),
        "Perplexity": str(metrics_held["perplexity"])
    },
    {
        "Corpus": "3. Out-of-Domain Technical",
        "Total Bigrams": metrics_ood["evaluated_bigrams"],
        "Seen": metrics_ood["seen_bigrams"],
        "Unseen": metrics_ood["unseen_bigrams"],
        "Coverage": f"{metrics_ood['coverage']:.1%}",
        "Avg Log-Prob": str(metrics_ood["average_log_probability"]),
        "Perplexity": str(metrics_ood["perplexity"])
    }
]

summary_df = pd.DataFrame(summary_data)
print("Intrinsic Evaluation Comparison Summary:")
display(summary_df)

---

## 🎓 Summary & Key Insights

Congratulations on completing the statistical bigram language model walkthrough! Here are the core conceptual takeaways:

1. **Discrete State Transitions (Markov Chain)**:
   A bigram model is simply a discrete Markov chain over the vocabulary. The next word is conditioned entirely on the current word $P(w_i \mid w_{i-1})$.
2. **Maximum Likelihood Estimation**:
   MLE computes optimal normalized empirical counts. While exact and interpretable, it assigns zero probability to unobserved transitions.
3. **Temperature Control**:
   The temperature parameter $T$ controls entropy across sampling. The exact same mechanism is used in modern Large Language Models (LLMs) such as GPT-4 and Gemini.
4. **The Zero-Frequency Problem & Smoothing**:
   Unsmoothed n-gram models fail on unseen transitions ($PP = \infty$). This directly motivates:
   - **Smoothing Techniques**: Laplace (Add-1), Good-Turing, or Kneser-Ney smoothing.
   - **Higher-Order N-grams**: Trigram ($n=3$) models with backoff or interpolation.
   - **Neural Language Models**: Using continuous dense vector embeddings (Word2Vec, Transformers) where semantically similar words share probability mass even if never seen together during training!

---

### Resources & Next Steps
- 📖 Explore the full codebase and documentation in the [simple-bigram-model GitHub Repository](https://github.com/thisiskamaljoshi/simple-bigram-model).
- 🧪 Train on the 50 MB TinyStories corpus using `python main.py train`.
- 🤝 Check out the [Contributing Guide](https://github.com/thisiskamaljoshi/simple-bigram-model/blob/main/CONTRIBUTING.md) to add Laplace smoothing or trigram modeling!